In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [0]:
dbutils.library.restartPython()

# 07 One-Click LexAI Runner (Final Stable)

Run cells top-to-bottom once.

Design goals:
- No hard failures across cells (graceful skip if prerequisite is missing)
- Databricks workspace notebook path handling via adapter fallback
- Correct browser links (driver-proxy), not localhost links
- Optional non-blocking Streamlit startup  Repos


In [0]:
# CELL 0.5: Optional Python restart (disabled by default)
# Set to True only when dependency stack is broken.
FORCE_PYTHON_RESTART = False

if FORCE_PYTHON_RESTART:
    print("[CELL 0.5] Restarting Python runtime...")
    dbutils.library.restartPython()
else:
    print("[CELL 0.5] Skipping restart (one-click mode).")


[CELL 0.5] Skipping restart (one-click mode).


In [0]:
# CELL 1: Runtime Flags
AUTO_INSTALL_MISSING = True
RUN_SMOKE_TEST = True

# Default to Streamlit-first on serverless for stability.
START_FASTAPI = False
START_STREAMLIT = True

# Try to auto-open Streamlit/FastAPI link in a new browser tab from notebook output.
AUTO_OPEN_UI_TAB = True

# On Databricks serverless/free, driver-proxy app URLs are often blocked.
# Keep False for stable runs. Set True only on compatible all-purpose compute.
ENABLE_FASTAPI_UI_ON_SERVERLESS = False

# Only use this if import stack is broken and you accept manual restart+rerun.
FORCE_REPAIR_IMPORT_STACK = False

FASTAPI_PORT = 8765
STREAMLIT_PORT = 8501

# Optional override if org id auto-detection fails (from URL ?o=<org_id>)
ORG_ID_OVERRIDE = ""

# Optional override for repo root. Keep empty for auto-detection.
REPO_DIR_OVERRIDE = ""

# Keep services alive after Run All. Set True only when you want to stop at the end.
STOP_SERVERS_AT_END = False

SMOKE_TEST_QUERIES = [
    "Penalty for not wearing helmet in short within 120 words",
    "What does section 129 say in detail within 180 words",
]

# Shared runtime flags
REPO_OK = False
DEPS_OK = False
ENGINE_READY = False
API_READY = False
STREAMLIT_READY = False

print("[CELL 1] Flags loaded")
print({
    "AUTO_INSTALL_MISSING": AUTO_INSTALL_MISSING,
    "RUN_SMOKE_TEST": RUN_SMOKE_TEST,
    "START_FASTAPI": START_FASTAPI,
    "START_STREAMLIT": START_STREAMLIT,
    "AUTO_OPEN_UI_TAB": AUTO_OPEN_UI_TAB,
    "ENABLE_FASTAPI_UI_ON_SERVERLESS": ENABLE_FASTAPI_UI_ON_SERVERLESS,
    "FORCE_REPAIR_IMPORT_STACK": FORCE_REPAIR_IMPORT_STACK,
    "FASTAPI_PORT": FASTAPI_PORT,
    "STREAMLIT_PORT": STREAMLIT_PORT,
    "ORG_ID_OVERRIDE": ORG_ID_OVERRIDE,
    "STOP_SERVERS_AT_END": STOP_SERVERS_AT_END,
    "REPO_DIR_OVERRIDE": REPO_DIR_OVERRIDE,
})


[CELL 1] Flags loaded
{'AUTO_INSTALL_MISSING': True, 'RUN_SMOKE_TEST': True, 'START_FASTAPI': True, 'START_STREAMLIT': False, 'FORCE_REPAIR_IMPORT_STACK': False, 'FASTAPI_PORT': 8765, 'STREAMLIT_PORT': 8501, 'ORG_ID_OVERRIDE': '7474658388963127', 'STOP_SERVERS_AT_END': False, 'REPO_DIR_OVERRIDE': '/Workspace/Repos/kumarangad54706@gmail.com/lexai-legal-rag-platform'}


In [0]:
# CELL 2: Resolve repo path safely
import os
import sys
from pathlib import Path
from datetime import datetime


def log(msg: str):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")


def _repo_has_required_files(repo_dir: Path) -> bool:
    return (
        (repo_dir / "apps" / "fastapi_app.py").exists()
        and (repo_dir / "apps" / "lexai06_notebook_adapter.py").exists()
    )


def _safe_walk_for_repo(root: Path):
    skip_dirs = {"__pycache__", ".git", ".ipynb_checkpoints"}

    def _onerror(_err):
        return None

    for dirpath, dirnames, _ in os.walk(root, topdown=True, onerror=_onerror):
        dirnames[:] = [d for d in dirnames if d not in skip_dirs]
        p = Path(dirpath)
        try:
            if _repo_has_required_files(p):
                return p
        except Exception:
            continue
    return None


def _context_repo_guess():
    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        nb_path = ctx.notebookPath().get()  # /Users/<email>/<repo>/notebooks/...
        if not nb_path:
            return None
        pp = Path(nb_path)
        ws_repo = Path("/Workspace") / Path(*pp.parent.parent.parts[1:])
        if ws_repo.exists() and _repo_has_required_files(ws_repo):
            return ws_repo
    except Exception:
        pass
    return None


def resolve_repo_dir() -> Path:
    if REPO_DIR_OVERRIDE and str(REPO_DIR_OVERRIDE).strip():
        p = Path(REPO_DIR_OVERRIDE.strip())
        if p.exists() and _repo_has_required_files(p):
            return p

    cwd = Path(os.getcwd()).resolve()
    for cand in [cwd] + list(cwd.parents):
        try:
            if _repo_has_required_files(cand):
                return cand
        except Exception:
            continue

    g = _context_repo_guess()
    if g is not None:
        return g

    for root in [Path("/Workspace/Repos"), Path("/Workspace/Users"), Path("/Workspace")]:
        if not root.exists():
            continue
        hit = _safe_walk_for_repo(root)
        if hit is not None:
            return hit

    return Path(os.getcwd()).resolve()


REPO_DIR = resolve_repo_dir()
if _repo_has_required_files(REPO_DIR):
    REPO_OK = True
    os.chdir(REPO_DIR)
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    log(f"Repo root: {REPO_DIR}")
    print("[CELL 2] REPO_OK=True")
else:
    REPO_OK = False
    print("[CELL 2] REPO_OK=False; unresolved repo path:", REPO_DIR)
    print("[CELL 2] Set REPO_DIR_OVERRIDE to your repo path and rerun from Cell 1.")


[15:29:48] Repo root: /Workspace/Repos/kumarangad54706@gmail.com/lexai-legal-rag-platform
[CELL 2] REPO_OK=True


In [0]:
# CELL 3: Dependency preflight - Databricks serverless-safe
import os
import sys
import importlib
import subprocess

DEPS_OK = False

if not REPO_OK:
    print("[CELL 3] Skipped: REPO_OK=False")
else:
    print("[CELL 3] Checking dependencies...")

    # Reduce optional-backend import issues in serverless sessions.
    os.environ.setdefault("TRANSFORMERS_NO_TF", "1")
    os.environ.setdefault("TRANSFORMERS_NO_FLAX", "1")
    os.environ.setdefault("USE_TF", "0")
    os.environ.setdefault("USE_FLAX", "0")
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

    required = [
        ("fastapi", "fastapi"),
        ("uvicorn", "uvicorn"),
        ("requests", "requests"),
        ("databricks.sdk", "databricks-sdk"),
        ("mlflow", "mlflow"),
        ("numpy", "numpy"),
        ("sentence_transformers", "sentence-transformers"),
        ("streamlit", "streamlit"),
    ]

    missing_pkgs = []
    for mod, pkg in required:
        try:
            importlib.import_module(mod)
        except Exception:
            missing_pkgs.append(pkg)

    # Keep transformer stack explicit for stability when sentence-transformers loads.
    extra_runtime_pkgs = ["transformers>=4.30.0", "accelerate>=0.20.0", "typing_extensions>=4.6.0", "streamlit>=1.36"]

    if missing_pkgs:
        print("[CELL 3] Missing packages:", missing_pkgs)
        if AUTO_INSTALL_MISSING:
            cmd = [sys.executable, "-m", "pip", "install", "-q"] + missing_pkgs + extra_runtime_pkgs
            try:
                print("[CELL 3] Installing missing packages...")
                subprocess.check_call(cmd)
                print("[CELL 3] Install completed")
            except Exception as e:
                print("[CELL 3] Install failed:", e)
        else:
            print("[CELL 3] AUTO_INSTALL_MISSING=False")

    # Clear possibly half-initialized imports from previous failed attempts.
    for k in list(sys.modules.keys()):
        if k.startswith(("accelerate", "transformers", "sentence_transformers", "jax", "flax", "tensorflow", "keras")):
            del sys.modules[k]

    try:
        import fastapi  # noqa: F401
        import uvicorn  # noqa: F401
        import requests  # noqa: F401
        import databricks.sdk  # noqa: F401
        import mlflow  # noqa: F401
        import numpy  # noqa: F401
        import sentence_transformers  # noqa: F401
        import streamlit  # noqa: F401
        DEPS_OK = True
        print("[CELL 3] DEPS_OK=True")
    except Exception as e:
        print("[CELL 3] Final dependency validation failed:", e)
        DEPS_OK = False

        if FORCE_REPAIR_IMPORT_STACK:
            print("[CELL 3] FORCE_REPAIR_IMPORT_STACK=True -> running repair install")
            repair_cmd = [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--upgrade",
                "typing_extensions>=4.6.0",
                "accelerate>=0.20.0",
                "transformers>=4.30.0",
                "sentence-transformers>=2.6.0",
                "streamlit>=1.36",
                "databricks-sdk",
            ]
            try:
                subprocess.check_call(repair_cmd)
                print("[CELL 3] Repair install done. If imports still fail, restart Python and rerun from Cell 1.")
            except Exception as e2:
                print("[CELL 3] Repair install failed:", e2)


[CELL 3] Checking dependencies...
[CELL 3] DEPS_OK=True


In [0]:
# CELL 4: Spark context and browser-link base
from pyspark.sql import SparkSession
import json
import re

cluster_id = "unknown"
org_id = "unknown"
workspace_url = "unknown"
compute_mode = "serverless_or_unknown"
DRIVER_PROXY_BASE = ""
DRIVER_PROXY_BASES = []
DRIVER_PROXY_SUPPORTED = False

try:
    spark = SparkSession.getActiveSession() or SparkSession.builder.getOrCreate()
    print("[CELL 4] Spark session ready:", bool(spark))
except Exception as e:
    spark = None
    print("[CELL 4] Spark unavailable:", e)


def _opt_to_str(opt):
    try:
        if hasattr(opt, "isDefined") and opt.isDefined():
            return str(opt.get())
    except Exception:
        pass
    try:
        return str(opt.get())
    except Exception:
        pass
    return ""


def _ctx_tag(ctx, key: str):
    try:
        tags = ctx.tags()
        return _opt_to_str(tags.get(key))
    except Exception:
        pass
    try:
        tags = ctx.tags()
        return str(tags.apply(key))
    except Exception:
        pass
    return ""


def _ctx_json(ctx):
    try:
        raw = _opt_to_str(ctx.toJson()) or str(ctx.toJson())
        if raw and raw.strip().startswith("{"):
            return json.loads(raw)
    except Exception:
        pass
    return {}


def _spark_conf_get(key: str):
    try:
        if spark is None:
            return ""
        v = spark.conf.get(key)
        return str(v) if v is not None else ""
    except Exception:
        return ""


def _first_non_empty(values):
    for v in values:
        s = str(v).strip()
        if s and s.lower() not in {"none", "null", "unknown"}:
            return s
    return ""


def _host_only(s: str):
    s = str(s or "").strip()
    if not s:
        return ""
    s = s.replace("https://", "").replace("http://", "")
    s = s.split("/")[0]
    return s.strip()


def _digits_only(s: str):
    s = str(s or "")
    m = re.search(r"(\d{6,})", s)
    return m.group(1) if m else ""


ctx = None
try:
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
except Exception:
    pass

ctxj = _ctx_json(ctx) if ctx else {}
ctx_tags = ctxj.get("tags", {}) if isinstance(ctxj, dict) else {}

cluster_id = _first_non_empty([
    _ctx_tag(ctx, "clusterId"),
    ctx_tags.get("clusterId", ""),
    _spark_conf_get("spark.databricks.clusterUsageTags.clusterId"),
    _spark_conf_get("spark.databricks.cluster.id"),
]) or "unknown"

org_id = _first_non_empty([
    str(ORG_ID_OVERRIDE).strip() if str(ORG_ID_OVERRIDE).strip() else "",
    _ctx_tag(ctx, "orgId"),
    _ctx_tag(ctx, "workspaceId"),
    ctx_tags.get("orgId", ""),
    ctx_tags.get("workspaceId", ""),
    _spark_conf_get("spark.databricks.clusterUsageTags.orgId"),
    _spark_conf_get("spark.databricks.workspace.id"),
    _spark_conf_get("spark.databricks.workspaceId"),
])
org_id = _digits_only(org_id) or (org_id if org_id else "unknown")

compute_mode = _first_non_empty([
    _ctx_tag(ctx, "clusterSource"),
    ctx_tags.get("clusterSource", ""),
    _spark_conf_get("spark.databricks.clusterUsageTags.clusterSource"),
]) or "serverless_or_unknown"

ws_from_ctx = _host_only(_opt_to_str(ctx.browserHostName()) if ctx else "")
ws_api = _host_only(_opt_to_str(ctx.apiUrl()) if ctx else "")
workspace_url = _first_non_empty([
    ws_from_ctx,
    ws_api,
    _host_only(_spark_conf_get("spark.databricks.workspaceUrl")),
    _host_only(ctx_tags.get("browserHostName", "")),
]) or "unknown"

DRIVER_PROXY_BASES = []
if workspace_url != "unknown" and org_id != "unknown" and cluster_id != "unknown":
    raw_bases = [
        f"https://{workspace_url}/driver-proxy/o/{org_id}/{cluster_id}",
    ]
    seen = set()
    for b in raw_bases:
        if b not in seen:
            seen.add(b)
            DRIVER_PROXY_BASES.append(b)

DRIVER_PROXY_BASE = DRIVER_PROXY_BASES[0] if DRIVER_PROXY_BASES else ""
DRIVER_PROXY_SUPPORTED = bool(DRIVER_PROXY_BASES)

print("[CELL 4] cluster_id:", cluster_id)
print("[CELL 4] org_id:", org_id)
print("[CELL 4] workspace_url:", workspace_url)
print("[CELL 4] compute_mode:", compute_mode)
print("[CELL 4] DRIVER_PROXY_SUPPORTED:", DRIVER_PROXY_SUPPORTED)
if DRIVER_PROXY_BASES:
    print("[CELL 4] DRIVER_PROXY_BASE candidates (browser-safe):")
    for b in DRIVER_PROXY_BASES:
        print(" -", b)
else:
    print("[CELL 4] DRIVER_PROXY_BASE: unavailable")


[CELL 4] Spark session ready: True


2026-03-02 15:30:00,491 11952 ERROR _handle_rpc_error GRPC Error received
Traceback (most recent call last):
  File "/databricks/python/lib/python3.10/site-packages/pyspark/sql/connect/client/core.py", line 1724, in config
    resp = self._stub.Config(req, metadata=self.metadata())
  File "/databricks/python/lib/python3.10/site-packages/grpc/_interceptor.py", line 277, in __call__
    response, ignored_call = self._with_call(
  File "/databricks/python/lib/python3.10/site-packages/grpc/_interceptor.py", line 332, in _with_call
    return call.result(), call
  File "/databricks/python/lib/python3.10/site-packages/grpc/_channel.py", line 439, in result
    raise self
  File "/databricks/python/lib/python3.10/site-packages/grpc/_interceptor.py", line 315, in continuation
    response, call = self._thunk(new_method).with_call(
  File "/databricks/python/lib/python3.10/site-packages/grpc/_channel.py", line 1193, in with_call
    return _end_unary_response_blocking(state, call, True, None)
 

[CELL 4] cluster_id: 0302-152820-dutz1yp9-v2n
[CELL 4] org_id: 7474658388963127
[CELL 4] workspace_url: dbc-afb2e98d-d930.cloud.databricks.com
[CELL 4] compute_mode: unknown
[CELL 4] DRIVER_PROXY_SUPPORTED: True
[CELL 4] DRIVER_PROXY_BASE: https://dbc-afb2e98d-d930.cloud.databricks.com/driver-proxy/o/7474658388963127/0302-152820-dutz1yp9-v2n


In [0]:
# CELL 4.5: Optional repair marker
if FORCE_REPAIR_IMPORT_STACK and not DEPS_OK:
    print("[CELL 4.5] Repair mode active and DEPS_OK=False.")
    print("[CELL 4.5] If you ran repair installs, run dbutils.library.restartPython() and rerun from Cell 1.")
else:
    print("[CELL 4.5] No repair action needed.")


[CELL 4.5] No repair action needed.


In [0]:
# CELL 5: Initialize notebook-06 engine through adapter
import os
from pathlib import Path
import importlib

ENGINE_READY = False
status = {}
last_err = None

print("[CELL 5] REPO_OK:", REPO_OK)
print("[CELL 5] DEPS_OK:", DEPS_OK)
print("[CELL 5] REPO_DIR:", REPO_DIR)

if not REPO_OK:
    print("[CELL 5] Cannot proceed: REPO_OK=False")
elif not DEPS_OK:
    print("[CELL 5] Cannot proceed: DEPS_OK=False")
else:
    import apps.lexai06_notebook_adapter as _adapter
    importlib.reload(_adapter)
    NotebookEngine = _adapter.NotebookEngine

    repo_dir = Path(REPO_DIR)

    # Prefer live notebook 06 first (latest logic), then snapshot fallback for serverless stability.
    snapshot_json = repo_dir / "apps" / "notebook_06_snapshot.json"
    snapshot_ipynb = repo_dir / "apps" / "notebook_06_snapshot.ipynb"
    snapshot_obj = repo_dir / "apps" / "notebook_06_snapshot"
    notebook_ipynb = repo_dir / "notebooks" / "06_High-precision_QA_Legal_Reasoning_Engine.ipynb"
    notebook_obj = repo_dir / "notebooks" / "06_High-precision_QA_Legal_Reasoning_Engine"

    candidates = [
        notebook_obj,
        notebook_ipynb,
        snapshot_obj,
        snapshot_ipynb,
        snapshot_json,
    ]

    # Add workspace-style paths from notebook context (/Repos/...) for export-based fallback.
    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        ws_nb_path = ctx.notebookPath().get()  # /Repos/<email>/<repo>/notebooks/07_...
        ws_nb_dir = Path(ws_nb_path).parent
        candidates.extend([
            ws_nb_dir / "06_High-precision_QA_Legal_Reasoning_Engine",
            ws_nb_dir.parent / "apps" / "notebook_06_snapshot",
            ws_nb_dir.parent / "apps" / "notebook_06_snapshot.json",
        ])
    except Exception:
        pass

    # De-duplicate preserving order.
    unique_candidates = []
    seen = set()
    for c in candidates:
        cs = str(c)
        if cs in seen:
            continue
        seen.add(cs)
        unique_candidates.append(c)

    print("[CELL 5] Notebook candidates:")
    for c in unique_candidates:
        try:
            print(" -", c, "exists=", Path(c).exists())
        except Exception:
            print(" -", c, "exists=ERROR")

    for cand in unique_candidates:
        try:
            os.environ["LEXAI06_NOTEBOOK_PATH"] = str(cand)
            engine = NotebookEngine(notebook_path=Path(str(cand)))
            status = engine.initialize()
            ENGINE_READY = bool(status.get("ready", False))
            if ENGINE_READY:
                print(f"[CELL 5] Initialized using candidate: {cand}")
                break
            print(f"[CELL 5] Candidate returned ready=False: {cand}")
        except Exception as e:
            last_err = e
            print(f"[CELL 5] Candidate failed: {cand} -> {e}")

    if ENGINE_READY:
        print("[CELL 5] Engine initialized")
        for k, v in status.items():
            print(f"  - {k}: {v}")
    else:
        print("[CELL 5] Engine not ready. Last error:", last_err)

print("[CELL 5] ENGINE_READY =", ENGINE_READY)


[CELL 5] REPO_OK: True
[CELL 5] DEPS_OK: True
[CELL 5] REPO_DIR: /Workspace/Repos/kumarangad54706@gmail.com/lexai-legal-rag-platform
[CELL 5] Notebook candidates:
 - /Workspace/Repos/kumarangad54706@gmail.com/lexai-legal-rag-platform/notebooks/06_High-precision_QA_Legal_Reasoning_Engine exists= True
 - /Workspace/Repos/kumarangad54706@gmail.com/lexai-legal-rag-platform/notebooks/06_High-precision_QA_Legal_Reasoning_Engine.ipynb exists= False
 - /Workspace/Repos/kumarangad54706@gmail.com/lexai-legal-rag-platform/apps/notebook_06_snapshot exists= True
 - /Workspace/Repos/kumarangad54706@gmail.com/lexai-legal-rag-platform/apps/notebook_06_snapshot.ipynb exists= False
 - /Workspace/Repos/kumarangad54706@gmail.com/lexai-legal-rag-platform/apps/notebook_06_snapshot.json exists= True
 - /Repos/kumarangad54706@gmail.com/lexai-legal-rag-platform/notebooks/06_High-precision_QA_Legal_Reasoning_Engine exists= False
 - /Repos/kumarangad54706@gmail.com/lexai-legal-rag-platform/apps/notebook_06_snaps

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

[15:30:18] Using endpoint backend: databricks-meta-llama-3-3-70b-instruct
[15:30:27] Loaded lexical artifacts from Delta for signature=312b53f785d939bab1c3.
--- Runtime Status ---
Data signature: 312b53f785d939bab1c3
Embedding source: /Volumes/workspace/legal_data/vector_db_test/legal_embeddings_delta
Records loaded: 5194
Embedding dim: 384
Avg doc len: 183.25
Vocabulary size: 13196
Lexical source: delta_artifact
Embedder: sentence-transformers/all-MiniLM-L6-v2
Reranker: cross-encoder/ms-marco-MiniLM-L-6-v2
LLM backend: endpoint (databricks-meta-llama-3-3-70b-instruct)
[CELL 5] Initialized using candidate: /Workspace/Repos/kumarangad54706@gmail.com/lexai-legal-rag-platform/notebooks/06_High-precision_QA_Legal_Reasoning_Engine
[CELL 5] Engine initialized
  - ready: True
  - error: 
  - notebook_path: /Workspace/Repos/kumarangad54706@gmail.com/lexai-legal-rag-platform/notebooks/06_High-precision_QA_Legal_Reasoning_Engine
  - resolved_notebook_path: /tmp/lexai06_exported_from_workspace.ip

In [0]:
# CELL 6: Smoke test
if RUN_SMOKE_TEST and ENGINE_READY:
    print("[CELL 6] Running smoke tests...")
    for idx, q in enumerate(SMOKE_TEST_QUERIES, start=1):
        print("=" * 90)
        print(f"[{idx}] Query: {q}")
        out = engine.answer_query(q)
        print("Mode:", out.get("mode"))
        print("Source:", out.get("source"))
        print("Confidence:", out.get("confidence"))
        print("Sections:", out.get("sections", []))
        print("Citations:", out.get("citations", [])[:5])
        print("Latency:", out.get("latency_ms", {}))
        print("Answer:")
        print(out.get("answer", ""))
    print("[CELL 6] Smoke tests done")
else:
    print("[CELL 6] Skipped (RUN_SMOKE_TEST or ENGINE_READY condition not met).")


[CELL 6] Running smoke tests...
[1] Query: Penalty for not wearing helmet in short within 120 words
Mode: rule_based
Source: traffic_rules
Confidence: top_score=0.617, avg_lex=0.732, coverage=0.444, normative_hits=5, judgment_hits=0, candidates=2512, shortlist=140, route=statute_strict
Sections: ['177', '178', '179', '180', '181', '182', '183', '184', '186', '189', '190', '191', '129']
Citations: ['Motor Vehicles Act 1988 - Section Chapter V', 'Motor Vehicles Act 1988 - Section Chapter VII', 'Motor Vehicle Ammendment Act 2019 - Section Chapter XI', 'Motor Vehicles Act 1988 - Section Chapter VI', 'Central Motor Vehicle Rules 1989 - Section Chapter VI']
Latency: {'candidate_fetch_ms': 3.99, 'routing_ms': 68.92, 'lexical_ms': 138.54, 'embed_ms': 62.65, 'dense_ms': 7.17, 'rrf_ms': 0.21, 'rerank_ms': 1216.87, 'retrieve_total_ms': 1502.42, 'generation_ms': 0.0, 'total_ms': 1504.22}
Answer:
Law:
Section 129 of the Motor Vehicles Act requires riders to wear protective headgear while riding two

In [0]:
# CELL 7: Start FastAPI and print exact browser links
import os
import sys
import subprocess
import time
from pathlib import Path
import requests

API_READY = False
API_HEALTH_VERIFIED = False
API_BROWSER_HEALTH_URL = ""
API_BROWSER_DOCS_URL = ""

print("[CELL 7] START_FASTAPI =", START_FASTAPI)
print("[CELL 7] ENGINE_READY =", ENGINE_READY)

if START_FASTAPI and ENGINE_READY:
    cm = (compute_mode or "").lower()
    serverless_like = ("serverless" in cm) or (cm in {"unknown", "serverless_or_unknown"})

    if serverless_like and (not ENABLE_FASTAPI_UI_ON_SERVERLESS):
        print("[CELL 7] FastAPI URL mode disabled on serverless/unknown compute.")
        print("[CELL 7] Streamlit direct-engine mode will be started in Cell 9.")
        API_READY = False
        API_HEALTH_VERIFIED = False
    else:
        FASTAPI_PROC = globals().get("FASTAPI_PROC")
        os.environ["NO_PROXY"] = "127.0.0.1,localhost,0.0.0.0"
        os.environ["no_proxy"] = "127.0.0.1,localhost,0.0.0.0"

        sess = requests.Session()
        sess.trust_env = False

        def _health(port: int, timeout_sec: int = 3):
            for h in ["127.0.0.1", "localhost", "0.0.0.0"]:
                try:
                    r = sess.get(f"http://{h}:{port}/health", timeout=timeout_sec)
                    if r.status_code == 200:
                        try:
                            body = r.json()
                        except Exception:
                            body = {}
                        if isinstance(body, dict) and ("ok" in body) and ("status" in body):
                            return True, body
                except Exception:
                    pass
            return False, {}

        def _wait(port: int, seconds: int = 30):
            t0 = time.time()
            while (time.time() - t0) < seconds:
                ok, body = _health(port, timeout_sec=2)
                if ok:
                    return True, body
                time.sleep(1)
            return False, {}

        ports = [int(FASTAPI_PORT), int(FASTAPI_PORT) + 1, int(FASTAPI_PORT) + 2, 8765, 8766, 8787]
        tried = []

        for p in ports:
            if p in tried:
                continue
            tried.append(p)

            ok0, _ = _health(p, timeout_sec=2)
            if ok0:
                FASTAPI_PORT = p
                API_READY = True
                API_HEALTH_VERIFIED = True
                print(f"[CELL 7] Reusing existing API on port {FASTAPI_PORT}")
                break

            log_path = Path('/tmp') / f'lexai_fastapi_{p}.log'
            lf = None
            try:
                lf = open(log_path, 'w', encoding='utf-8')
            except Exception:
                pass

            cmd = [
                sys.executable, '-m', 'uvicorn', 'apps.fastapi_app:app',
                '--host', '0.0.0.0', '--port', str(p), '--workers', '1', '--log-level', 'warning'
            ]
            env = os.environ.copy()
            env['PYTHONPATH'] = f"{REPO_DIR}:{env.get('PYTHONPATH','')}"
            env['NO_PROXY'] = '127.0.0.1,localhost,0.0.0.0'
            env['no_proxy'] = '127.0.0.1,localhost,0.0.0.0'

            try:
                proc = subprocess.Popen(
                    cmd,
                    cwd=str(REPO_DIR),
                    env=env,
                    stdout=lf if lf is not None else subprocess.DEVNULL,
                    stderr=lf if lf is not None else subprocess.DEVNULL,
                )
            except Exception as e:
                if lf is not None:
                    lf.close()
                print(f"[CELL 7] Launch failed on port {p}: {e}")
                continue

            ok1, _ = _wait(p, seconds=20)
            if ok1:
                FASTAPI_PORT = p
                API_READY = True
                API_HEALTH_VERIFIED = True
                globals()['FASTAPI_PROC'] = proc
                globals()['FASTAPI_LOG_PATH'] = str(log_path)
                print(f"[CELL 7] FastAPI started and verified on port {FASTAPI_PORT}")
                if lf is not None:
                    lf.close()
                break

            try:
                if proc.poll() is None:
                    proc.terminate()
            except Exception:
                pass
            if lf is not None:
                lf.close()
            print(f"[CELL 7] Port {p} not healthy")

        if API_READY and API_HEALTH_VERIFIED:
            print("[CELL 7] Local health check: OK")
            if DRIVER_PROXY_SUPPORTED:
                API_BROWSER_HEALTH_URL = f"{DRIVER_PROXY_BASE}/{FASTAPI_PORT}/health"
                API_BROWSER_DOCS_URL = f"{DRIVER_PROXY_BASE}/{FASTAPI_PORT}/docs"
                print("[CELL 7] Browser URL (health):", API_BROWSER_HEALTH_URL)
                print("[CELL 7] Browser URL (docs):", API_BROWSER_DOCS_URL)
                try:
                    displayHTML(f'<a href="{API_BROWSER_DOCS_URL}" target="_blank">Open FastAPI Docs</a>')
                except Exception:
                    pass
            else:
                print("[CELL 7] Driver proxy metadata unavailable.")
        else:
            print("[CELL 7] FastAPI URL mode unavailable in this session.")
            print("[CELL 7] Streamlit direct-engine mode will run in Cell 9.")

elif START_FASTAPI and not ENGINE_READY:
    print("[CELL 7] Skipped: ENGINE_READY=False. Fix Cell 5 first.")
else:
    print("[CELL 7] FastAPI step skipped (START_FASTAPI=False).")
    print("[CELL 7] Streamlit direct-engine mode will run in Cell 9.")


[CELL 7] START_FASTAPI = True
[CELL 7] ENGINE_READY = True
[CELL 7] FastAPI process is running on port 8765, but health probe is blocked/unverified.
[CELL 7] This serverless runtime is not exposing a usable driver-proxy app endpoint.
[CELL 7] Falling back to direct-engine mode for reliable execution.
[CELL 7] Local health check: UNAVAILABLE
[CELL 7] Browser/API URL disabled in this session. Use Cell 8 direct-engine fallback.


In [0]:
# CELL 8: FastAPI smoke call (with direct-engine fallback)
import requests

query_text = "What is the penalty for not wearing a helmet?"

if API_READY and API_HEALTH_VERIFIED:
    try:
        h = requests.get(f"http://127.0.0.1:{FASTAPI_PORT}/health", timeout=30)
        print("[CELL 8] local /health status:", h.status_code)
        try:
            print(h.json())
        except Exception:
            print(h.text[:300])

        payload = {
            "query": query_text,
            "style": "short",
            "word_limit": 120,
        }
        r = requests.post(f"http://127.0.0.1:{FASTAPI_PORT}/v1/legal/answer", json=payload, timeout=180)
        print("[CELL 8] local /v1/legal/answer status:", r.status_code)
        try:
            body = r.json()
            print("[CELL 8] answer preview:", body.get("answer", "")[:500])
        except Exception:
            print("[CELL 8] raw response:", r.text[:500])

        if API_BROWSER_DOCS_URL:
            print("[CELL 8] Browser docs URL:", API_BROWSER_DOCS_URL)
    except Exception as e:
        print("[CELL 8] API call failed, switching to direct-engine fallback:", e)

if not (API_READY and API_HEALTH_VERIFIED):
    print("[CELL 8] Running direct engine fallback...")
    try:
        out = engine.answer_query(query_text + " in short within 120 words")
        print("[CELL 8] direct-engine answer preview:", out.get("answer", "")[:500])
        print("[CELL 8] sections:", out.get("sections", []))
        print("[CELL 8] citations:", out.get("citations", [])[:5])
        print("[CELL 8] source:", out.get("source"))
    except Exception as e2:
        print("[CELL 8] direct-engine fallback failed:", e2)


[CELL 8] Running direct engine fallback...
[CELL 8] direct-engine answer preview: Law:
Section 129 of the Motor Vehicles Act requires riders to wear protective headgear while riding two-wheelers in public places.

Penalty:
Violation may attract enforcement under Section 177/194D-style traffic penalty provisions, often including monetary fine and possible licence-related consequences depending on state notifications.

Why this rule exists:
Helmet compliance reduces severe head injuries and road fatalities.

Advice:
Use a BIS-approved helmet with strap fastened and follow chall
[CELL 8] sections: ['177', '178', '179', '180', '181', '182', '183', '184', '186', '189', '190', '191', '129']
[CELL 8] citations: ['Motor Vehicles Act 1988 - Section Chapter V', 'Motor Vehicles Act 1988 - Section Chapter VII', 'Motor Vehicle Ammendment Act 2019 - Section Chapter XI', 'Motor Vehicles Act 1988 - Section Chapter VI', 'Central Motor Vehicle Rules 1989 - Section Chapter VI']
[CELL 8] source: traffic_r

In [0]:
# CELL 9: Optional Streamlit start (supports direct-engine mode)
import os
import sys
import subprocess
import time
import requests
import importlib
import socket
import re

STREAMLIT_READY = False
STREAMLIT_HEALTH_VERIFIED = False
STREAMLIT_BROWSER_URL = ""
STREAMLIT_PROC = globals().get("STREAMLIT_PROC")
STREAMLIT_NATIVE_URLS = globals().get("STREAMLIT_NATIVE_URLS", {})


def _ensure_streamlit():
    try:
        st = importlib.import_module("streamlit")
        print(f"[CELL 9] streamlit available: version={getattr(st, '__version__', 'unknown')}")
        return True
    except Exception:
        pass

    print("[CELL 9] streamlit not found in current Python env; installing...")
    if not AUTO_INSTALL_MISSING:
        print("[CELL 9] AUTO_INSTALL_MISSING=False. Set it True in Cell 1 or install manually:")
        print("[CELL 9]   %pip install streamlit>=1.36")
        return False

    try:
        cmd = [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "streamlit>=1.36"]
        subprocess.check_call(cmd)
    except Exception as e:
        print(f"[CELL 9] streamlit install failed: {e}")
        print("[CELL 9] Manual fallback: %pip install streamlit>=1.36")
        return False

    importlib.invalidate_caches()
    try:
        if "streamlit" in sys.modules:
            del sys.modules["streamlit"]
        st = importlib.import_module("streamlit")
        print(f"[CELL 9] streamlit installation verified: version={getattr(st, '__version__', 'unknown')}")
        return True
    except Exception as e:
        print(f"[CELL 9] streamlit import still failing after install: {e}")
        print(f"[CELL 9] Active interpreter: {sys.executable}")
        return False


def _no_proxy_session():
    s = requests.Session()
    s.trust_env = False
    return s


def _port_open(port: int, timeout_sec: float = 1.0) -> bool:
    try:
        with socket.create_connection(("127.0.0.1", int(port)), timeout=timeout_sec):
            return True
    except Exception:
        return False


def _streamlit_health(port: int, timeout_sec: int = 2):
    sess = _no_proxy_session()
    urls = [
        f"http://127.0.0.1:{port}/_stcore/health",
        f"http://localhost:{port}/_stcore/health",
    ]
    for u in urls:
        try:
            r = sess.get(u, timeout=timeout_sec)
            if r.status_code == 200:
                return True
        except Exception:
            pass
    return False


def _streamlit_root_reachable(port: int, timeout_sec: int = 2):
    sess = _no_proxy_session()
    urls = [
        f"http://127.0.0.1:{port}/",
        f"http://localhost:{port}/",
    ]
    for u in urls:
        try:
            r = sess.get(u, timeout=timeout_sec, allow_redirects=False)
            if r.status_code in (200, 301, 302, 303, 307, 308):
                return True
        except Exception:
            pass
    return False


def _tail(path, lines=30):
    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            arr = f.read().splitlines()
        return "\n".join(arr[-lines:])
    except Exception:
        return ""


def _extract_streamlit_urls(log_path: str):
    out = {}
    try:
        if not log_path:
            return out
        txt = Path(log_path).read_text(encoding="utf-8", errors="ignore")
        for m in re.finditer(r"^\s*(Local|Network|External) URL:\s*(\S+)\s*$", txt, re.MULTILINE):
            out[m.group(1).lower()] = m.group(2).strip()
    except Exception:
        pass
    return out


def _wait_streamlit(port: int, log_path: str, timeout_sec: int = 75):
    t0 = time.time()
    while (time.time() - t0) < timeout_sec:
        if _streamlit_health(port, timeout_sec=2):
            return True, "health", _extract_streamlit_urls(log_path)
        # Fallback signal: process bound and root endpoint responding.
        if _port_open(port, timeout_sec=1.0) and _streamlit_root_reachable(port, timeout_sec=2):
            return True, "root", _extract_streamlit_urls(log_path)
        # Serverless fallback: trust Streamlit startup banner if URLs appear in logs.
        urls = _extract_streamlit_urls(log_path)
        if urls:
            return True, "log_banner", urls
        time.sleep(1)
    return False, "timeout", _extract_streamlit_urls(log_path)


def _stop_proc(proc, wait_sec: int = 8):
    if proc is None:
        return
    try:
        if proc.poll() is None:
            proc.terminate()
            t0 = time.time()
            while (time.time() - t0) < wait_sec:
                if proc.poll() is not None:
                    return
                time.sleep(0.5)
            proc.kill()
    except Exception:
        pass


def _resolve_proxy_bases():
    bases = []

    for b in globals().get("DRIVER_PROXY_BASES", []) or []:
        s = str(b).strip()
        # Keep only browser-safe app proxy (exclude driver-proxy-api -> 401 in browser).
        if s and "/driver-proxy/o/" in s:
            bases.append(s)

    base = str(globals().get("DRIVER_PROXY_BASE", "") or "").strip()
    if base and "/driver-proxy/o/" in base:
        bases.append(base)

    ws = str(globals().get("workspace_url", "") or "").strip()
    cid = str(globals().get("cluster_id", "") or "").strip()
    oid = str(globals().get("org_id", "") or "").strip()

    if ws and ws != "unknown" and cid and cid != "unknown" and oid and oid != "unknown":
        bases.append(f"https://{ws}/driver-proxy/o/{oid}/{cid}")

    out = []
    seen = set()
    for b in bases:
        if b not in seen:
            seen.add(b)
            out.append(b)
    return out


if START_STREAMLIT and ENGINE_READY:
    if not _ensure_streamlit():
        print("[CELL 9] Cannot start Streamlit because module is unavailable.")
    else:
        direct_mode = not (API_READY and API_HEALTH_VERIFIED)

        if STREAMLIT_PROC is not None and STREAMLIT_PROC.poll() is None:
            # Do not trust stale process handles after reruns/restarts.
            if _streamlit_health(int(STREAMLIT_PORT), timeout_sec=2):
                STREAMLIT_READY = True
                STREAMLIT_HEALTH_VERIFIED = True
                print(f"[CELL 9] Streamlit already running and healthy on port {STREAMLIT_PORT}")
            else:
                prev_log = str(globals().get("STREAMLIT_LOG_PATH", "") or "")
                prev_urls = _extract_streamlit_urls(prev_log)
                if prev_urls:
                    STREAMLIT_READY = True
                    STREAMLIT_HEALTH_VERIFIED = False
                    STREAMLIT_NATIVE_URLS = prev_urls
                    globals()["STREAMLIT_NATIVE_URLS"] = prev_urls
                    print(f"[CELL 9] Streamlit process already running on port {STREAMLIT_PORT} (verified by log banner).")
                else:
                    print(f"[CELL 9] Existing Streamlit process on port {STREAMLIT_PORT} is not healthy; restarting it.")
                    _stop_proc(STREAMLIT_PROC)
                    STREAMLIT_PROC = None
                    globals()["STREAMLIT_PROC"] = None

        if not STREAMLIT_READY:
            selected_port = int(STREAMLIT_PORT)
            started = False
            port_candidates = [selected_port, selected_port + 1, selected_port + 2, 8510, 8520, 8530]
            tried = set()

            for p in port_candidates:
                p = int(p)
                if p in tried:
                    continue
                tried.add(p)

                log_path = f"/tmp/lexai_streamlit_{p}.log"
                globals()["STREAMLIT_LOG_PATH"] = log_path
                try:
                    lf = open(log_path, "w", encoding="utf-8")
                except Exception:
                    lf = None

                proc = None
                try:
                    cmd = [
                        sys.executable,
                        "-m",
                        "streamlit",
                        "run",
                        "apps/streamlit_app.py",
                        "--server.port",
                        str(p),
                        "--server.address",
                        "0.0.0.0",
                        "--server.headless",
                        "true",
                    ]
                    env = os.environ.copy()
                    env["PYTHONPATH"] = f"{REPO_DIR}:{env.get('PYTHONPATH', '')}"
                    env["NO_PROXY"] = "127.0.0.1,localhost,0.0.0.0"
                    env["no_proxy"] = "127.0.0.1,localhost,0.0.0.0"
                    env["LEXAI_STREAMLIT_DIRECT_ENGINE"] = "1" if direct_mode else "0"
                    env["LEXAI_API_BASE_URL"] = f"http://127.0.0.1:{FASTAPI_PORT}"

                    proc = subprocess.Popen(
                        cmd,
                        cwd=str(REPO_DIR),
                        env=env,
                        stdout=lf if lf is not None else subprocess.DEVNULL,
                        stderr=lf if lf is not None else subprocess.DEVNULL,
                    )

                    time.sleep(3)
                    rc = proc.poll()
                    if rc is not None:
                        if lf is not None:
                            lf.close()
                        tail = _tail(log_path, lines=25)
                        print(f"[CELL 9] Streamlit exited immediately on port {p} (rc={rc})")
                        if tail:
                            print(f"[CELL 9] Log tail for port {p}:\n{tail}")
                        continue

                    ready, mode, native_urls = _wait_streamlit(p, log_path=log_path, timeout_sec=75)
                    if ready:
                        STREAMLIT_PROC = proc
                        STREAMLIT_PORT = int(p)
                        STREAMLIT_READY = True
                        STREAMLIT_HEALTH_VERIFIED = mode == "health"
                        STREAMLIT_NATIVE_URLS = native_urls or {}
                        globals()["STREAMLIT_PROC"] = STREAMLIT_PROC
                        globals()["STREAMLIT_NATIVE_URLS"] = STREAMLIT_NATIVE_URLS
                        started = True
                        mode_txt = "direct-engine" if direct_mode else "api"
                        probe_txt = "health" if STREAMLIT_HEALTH_VERIFIED else mode
                        print(
                            f"[CELL 9] Streamlit started on port {STREAMLIT_PORT} "
                            f"({mode_txt} mode, verified by {probe_txt})"
                        )
                        if lf is not None:
                            lf.close()
                        break

                    if lf is not None:
                        lf.close()
                    tail = _tail(log_path, lines=25)
                    print(f"[CELL 9] Port {p} did not become reachable in time; trying next port.")
                    if tail:
                        print(f"[CELL 9] Log tail for port {p}:\n{tail}")
                    _stop_proc(proc)
                except Exception as e:
                    if lf is not None:
                        lf.close()
                    print(f"[CELL 9] Start failed on port {p}: {e}")
                    _stop_proc(proc)
                    continue

            if not started:
                print("[CELL 9] Streamlit failed to start on tried ports.")

        if STREAMLIT_READY:
            # 1) Databricks proxy URL(s)
            proxy_bases = _resolve_proxy_bases()
            if proxy_bases:
                urls = [f"{b}/{STREAMLIT_PORT}/" for b in proxy_bases]
                STREAMLIT_BROWSER_URL = urls[0]
                print("[CELL 9] Browser URL candidates (Databricks driver-proxy):")
                for u in urls:
                    print(" -", u)
                try:
                    links = "<br>".join([f'<a href="{u}" target="_blank">{u}</a>' for u in urls])
                    html = links
                    if AUTO_OPEN_UI_TAB and STREAMLIT_BROWSER_URL:
                        html += f'<script>window.open("{STREAMLIT_BROWSER_URL}", "_blank");</script>'
                    displayHTML(html)
                except Exception:
                    pass
            else:
                print("[CELL 9] Proxy URL metadata is missing (workspace/org/cluster info).")

            # 2) Native Streamlit URLs from startup logs (may or may not be reachable externally)
            if STREAMLIT_NATIVE_URLS:
                print("[CELL 9] Native Streamlit URLs from process log:")
                for k in ["local", "network", "external"]:
                    if k in STREAMLIT_NATIVE_URLS:
                        print(f" - {k}: {STREAMLIT_NATIVE_URLS[k]}")

            if (not proxy_bases) and (not STREAMLIT_NATIVE_URLS):
                print("[CELL 9] Streamlit is running but no browser URL could be derived.")
                print("[CELL 9] Rerun Cell 4, or set ORG_ID_OVERRIDE in Cell 1 (value from workspace URL '?o=<org_id>').")
else:
    print("[CELL 9] Skipped (START_STREAMLIT or ENGINE_READY condition not met).")


[CELL 9] Skipped (START_STREAMLIT or API_READY condition not met).


In [0]:
# CELL 10: Stop helper
if not STOP_SERVERS_AT_END:
    print("[CELL 10] STOP_SERVERS_AT_END=False -> keeping FastAPI/Streamlit running.")
    print("[CELL 10] Set STOP_SERVERS_AT_END=True in Cell 1 and rerun Cell 10 when you want to stop services.")
else:
    # Stop subprocess-based FastAPI if used.
    if "FASTAPI_PROC" in globals() and globals().get("FASTAPI_PROC") is not None:
        proc = globals()["FASTAPI_PROC"]
        try:
            if proc.poll() is None:
                proc.terminate()
                print("[CELL 10] FastAPI subprocess termination requested")
            else:
                print("[CELL 10] FastAPI subprocess already stopped")
        except Exception as e:
            print("[CELL 10] FastAPI subprocess stop failed:", e)
    # Backward compatibility: stop thread-based server if present.
    elif "FASTAPI_SERVER" in globals() and globals().get("FASTAPI_SERVER") is not None:
        globals()["FASTAPI_SERVER"].should_exit = True
        print("[CELL 10] FastAPI thread stop requested")
    else:
        print("[CELL 10] FastAPI was not running")

    if "STREAMLIT_PROC" in globals() and globals().get("STREAMLIT_PROC") is not None:
        proc = globals()["STREAMLIT_PROC"]
        try:
            if proc.poll() is None:
                proc.terminate()
                print("[CELL 10] Streamlit process termination requested")
            else:
                print("[CELL 10] Streamlit process already stopped")
        except Exception as e:
            print("[CELL 10] Streamlit stop failed:", e)
    else:
        print("[CELL 10] Streamlit was not running")


[CELL 10] STOP_SERVERS_AT_END=False -> keeping FastAPI/Streamlit running.
[CELL 10] Set STOP_SERVERS_AT_END=True in Cell 1 and rerun Cell 10 when you want to stop services.
